In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "b2b-test-research"

In [2]:
from components.nodes import test_research_node

In [6]:
requirement = "The property must provide private meeting or event space suitable for at least 20 participants, and catering or banquet service for group events."

hotels = [
    {"name": "Bangkok Marriott Hotel Sukhumvit"},
    {"name": "ibis Styles Bangkok Sukhumvit Phra Khanong"},
    {"name": "ibis Bangkok Sukhumvit 4"},
    {"name": "Citin Sukhumvit 11 Nana Bangkok by Compass Hospitality"},
    {"name": "Centre Point Hotel Sukhumvit 10"},
    {"name": "Aloft by Marriott Bangkok Sukhumvit 11"},
    {"name": "HOP INN Bangkok Onnut Station"},
    {"name": "Avani Sukhumvit Bangkok"},
]

In [4]:
def _field(item, name):
    if isinstance(item, dict):
        return item.get(name)
    return getattr(item, name)


def show_result(company, result):
    print(company)
    print("sufficient:", result.get("sufficient"))
    print("reason:", result.get("sufficient_reason"))
    print("search_tool_call_count:", result.get("search_tool_call_count"))
    print("inspect_tool_call_count:", result.get("inspect_tool_call_count"))
    print("write_evidence_tool_call_count:", result.get("write_evidence_tool_call_count"))
    print("think_tool_call_count:", result.get("think_tool_call_count"))
    print(
        "total_tool_call_count:",
        (result.get("search_tool_call_count") or 0)
        + (result.get("inspect_tool_call_count") or 0)
        + (result.get("write_evidence_tool_call_count") or 0)
        + (result.get("think_tool_call_count") or 0),
    )
    print("queries:", result.get("search_queries_used"))
    print("additional_evidence_needed:", result.get("additional_evidence_needed"))
    print("verified:")
    for item in result.get("verified_evidence") or []:
        print(f"- {_field(item, 'claim')}")
        print(f"  {_field(item, 'url')}")
    print("failed:")
    for item in result.get("failed_evidence") or []:
        print(f"- {_field(item, 'claim')} ({_field(item, 'reason')})")
    print("verifier_judgments:")
    for item in result.get("verifier_judgments") or []:
        print(f"- support={_field(item, 'support')} | {_field(item, 'claim')}")
        print(f"  {_field(item, 'reason')}")
        passages = _field(item, "supporting_passages") or []
        if passages:
            print(f"  passages: {passages}")

In [7]:
hotel = hotels[0]
result = await test_research_node({
    "company": hotel["name"],
    "requirement": requirement,
})
show_result(hotel["name"], result)

Bangkok Marriott Hotel Sukhumvit
sufficient: True
reason: The Marriott events page (result_id call_qVpgw2INe0yjRtpdfd6EzFgB_1) explicitly lists 7 event rooms with banquet capacities well above 20 (examples: The Sethi Studio – banquet 50; Benchasiri Ballroom B – banquet 90; Benchasiri Ballroom A – banquet 200; Benchasiri Ballroom – banquet 300–450) and includes a 'Catering Services' section, verifying both private event space capacity and availability of catering/banquet services.
search_tool_call_count: 1
queries: ['Bangkok Marriott Hotel Sukhumvit meeting event space catering banquet 20 participants']
additional_evidence_needed: []
verified:
- Bangkok Marriott Hotel Sukhumvit has 7 event rooms and lists meeting rooms with banquet capacities such as The Sethi Studio (banquet 50), Benchasiri Ballroom B (banquet 90), Benchasiri Ballroom A (banquet 200) and Benchasiri Ballroom (banquet 300-450), demonstrating private event spaces suitable for at least 20 participants.
  https://www.marrio

In [ ]:
results = []
for hotel in hotels:
    out = await test_research_node({
        "company": hotel["name"],
        "requirement": requirement,
    })
    row = {"company": hotel["name"], **out}
    results.append(row)
    show_result(hotel["name"], out)
    print("---")

results